1. what is the equation for logistic regression that your classifier found from training years?

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('KO_weekly_return_volatility.csv')

train = df[df['Year'].isin([2021, 2022, 2023])]
test = df[df['Year'].isin([2024, 2025])]

X_train = train[['mean_return', 'volatility']].values
Y_train = train['decision'].values
X_test = test[['mean_return', 'volatility']].values
Y_test = test['decision'].values

# training logistic regression
lr = LogisticRegression()
lr.fit(X_train, Y_train)

# extracting the equation coefficients
w0 = lr.intercept_[0]
w1 = lr.coef_[0][0]
w2 = lr.coef_[0][1]

print(f'w0 (intercept) = {round(w0, 4)}')
print(f'w1 (mean_return) = {round(w1, 4)}')
print(f'w2 (volatility) = {round(w2, 4)}')
print(f'\nEquation: p(YES) = 1 / (1 + e^(-({round(w0,4)} + {round(w1,4)} * mean_return + {round(w2,4)} * volatility)))')

w0 (intercept) = -0.8417
w1 (mean_return) = 0.7214
w2 (volatility) = 0.2491

Equation: p(YES) = 1 / (1 + e^(-(-0.8417 + 0.7214 * mean_return + 0.2491 * volatility)))


2. what is the accuracy for testing years?

In [2]:
# checking accuracy on test data
test_accuracy = lr.score(X_test, Y_test)
print(f'Test accuracy: {round(test_accuracy * 100, 2)}%')

Test accuracy: 69.81%


3. compute the confusion matrix for testing years

In [3]:
from sklearn.metrics import confusion_matrix

# getting predictions for test data
predictions = lr.predict(X_test)

# confusion matrix
cm = confusion_matrix(Y_test, predictions, labels=['YES', 'NO'])
print('Confusion Matrix:')
print(f'                Predicted YES   Predicted NO')
print(f'Actual YES      {cm[0][0]}              {cm[0][1]}')
print(f'Actual NO       {cm[1][0]}              {cm[1][1]}')

Confusion Matrix:
                Predicted YES   Predicted NO
Actual YES      2              31
Actual NO       1              72


4. what is true positive rate (sensitivity or recall) and true negative rate (specificity) for testing years?

In [4]:
TP = cm[0][0]  # 2
FN = cm[0][1]  # 31
FP = cm[1][0]  # 1
TN = cm[1][1]  # 72

TPR = TP / (TP + FN)
TNR = TN / (TN + FP)

print(f'TPR (sensitivity): {round(TPR * 100, 2)}%')
print(f'TNR (specificity): {round(TNR * 100, 2)}%')

TPR (sensitivity): 6.06%
TNR (specificity): 98.63%


5. implement a trading strategy based on your labels for testing years and compare the performance with the "buy-and-hold" strategy. Which strategy results in a larger amount at the end of the testing period?

In [5]:
daily_data = pd.read_csv('KO_weekly_return_detailed.csv')
daily_test = daily_data[daily_data['Year'].isin([2024, 2025])].reset_index(drop=True)

test_reset = test.reset_index(drop=True)
test_reset['lr_prediction'] = predictions

# LOGISTIC REGRESSION TRADING STRATEGY
initial_balance = 100
lr_balance = initial_balance
position = None

for week_idx in range(len(test_reset)):
    current_week = test_reset.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']

    week_data = daily_test[
        (daily_test['Year'] == year) &
        (daily_test['Week_Number'] == week_num)
    ]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                lr_balance = lr_balance * (1 + row['Return'] / 100)

    if week_idx < len(test_reset) - 1:
        next_prediction = test_reset.iloc[week_idx + 1]['lr_prediction']
        if next_prediction == 'YES':
            position = 'invested'
        else:
            position = None

# BUY AND HOLD
bh_balance = initial_balance
for idx, row in daily_test.iterrows():
    if pd.notna(row['Return']):
        bh_balance = bh_balance * (1 + row['Return'] / 100)

print(f'Logistic Regression Strategy: ${lr_balance:.2f}')
print(f'Buy and Hold: ${bh_balance:.2f}')

Logistic Regression Strategy: $114.58
Buy and Hold: $126.14


Buy and Hold strategy results in a larger amount at the end of the testing period